# Exploratory Data Analysis


In [1]:
import numpy as np
import pandas as pd
import yfinance as yf
from market_ml.data_loader import DataLoader
import matplotlib.pyplot as plt
import plotly.express as px
from market_ml.config import * 
from market_ml.EDA import * 

# Data Loading 


In [2]:
dl = DataLoader(tickers=TICKERS, period="10y", interval="1d", file_name="stock_data.parquet")
market_data = dl.load_data()
sectors = market_data["Sector"].unique()


Data loaded for ['AAPL']


# Exploring the market data df

In [3]:
market_data.info()

<class 'pandas.DataFrame'>
RangeIndex: 2514 entries, 0 to 2513
Data columns (total 8 columns):
 #   Column  Non-Null Count  Dtype         
---  ------  --------------  -----         
 0   Date    2514 non-null   datetime64[ms]
 1   Ticker  2514 non-null   str           
 2   Sector  2514 non-null   str           
 3   Close   2514 non-null   float64       
 4   High    2514 non-null   float64       
 5   Low     2514 non-null   float64       
 6   Open    2514 non-null   float64       
 7   Volume  2514 non-null   int64         
dtypes: datetime64[ms](1), float64(4), int64(1), str(2)
memory usage: 191.6 KB


In [4]:
display(market_data.shape)
display(market_data['Ticker'].unique())
display(market_data['Sector'].value_counts())



(2514, 8)

<ArrowStringArray>
['AAPL']
Length: 1, dtype: str

Sector
Technology    2514
Name: count, dtype: int64

Summary of the dataframe by ticker

In [5]:
market_data.groupby("Ticker").agg(
    Rows=("Ticker", "size"),
    Start=("Date", "min"),
    End=("Date", "max")
)

,Rows,Start,End
Ticker,,,
AAPL,2514,2016-11-14,2026-11-13


Great, no obvious inbalance in the data 

# Daily Returns

In [6]:
market_data = calculate_returns(market_data)
market_data["Daily_Return"].describe()

count    2513.000000
mean        0.001238
std         0.018379
min        -0.128647
25%        -0.007263
50%         0.001172
75%         0.010209
max         0.153288
Name: Daily_Return, dtype: float64

In [7]:
#For each sector, plotting the daily returns of each of the stocks in that sector. 
for sector in sectors:
    sector_data = market_data[market_data["Sector"]==sector]
       
    fig = px.histogram(sector_data, x="Daily_Return", nbins=100, facet_col='Ticker',title=f'{sector} Daily Return Distributions' )
    fig.show()
    

#### Daily returns mostly centered around 0.0 but some large values in the tails, showing overall the daily returns are small with some rarer big changes. 

# Volatility

In [8]:
#Calculating a rolling 20 day volatility. 
market_data = calculate_rolling_volatility(market_data, window=20) 

In [9]:
market_data

Price,Date,Ticker,Sector,Close,High,Low,Open,Volume,Daily_Return,Volatility_20D
0,2016-11-14,AAPL,Technology,24.280031,24.762370,23.905645,24.739402,204702000,NaN,NaN
1,2016-11-15,AAPL,Technology,24.601593,24.732514,24.383393,24.477563,129058000,0.013244,NaN
2,2016-11-16,AAPL,Technology,25.263075,25.318201,24.484442,24.507410,235362000,0.026888,NaN
3,2016-11-17,AAPL,Technology,25.253891,25.345765,24.996645,25.221735,110528000,-0.000364,NaN
4,2016-11-18,AAPL,Technology,25.279156,25.389405,25.187283,25.201064,113715600,0.001000,NaN
...,...,...,...,...,...,...,...,...,...,...
2509,2026-11-09,AAPL,Technology,350.423737,352.062476,344.518220,344.738054,48753100,0.024301,0.016953
2510,2026-11-10,AAPL,Technology,348.245392,353.601307,346.196930,351.083232,29756100,-0.006216,0.017042
2511,2026-11-11,AAPL,Technology,355.359985,355.639772,349.064750,349.284585,34453500,0.020430,0.017565
2512,2026-11-12,AAPL,Technology,354.880005,359.369995,354.410004,357.809998,31875000,-0.001351,0.017548


In [10]:
#For each sector, plotting the rolling volalitly of each of the stocks in that sector. 
for sector in sectors:
    sector_data = market_data[market_data["Sector"]==sector]
    fig = px.line(sector_data, x="Date", y="Volatility_20D", color="Ticker", title=f' {sector} 20 Day Rolling Volatility')
    fig.update_layout(yaxis_title="Volatility", xaxis_title="Date")
    fig.show()
    

#### Rolling 20 day volatility varies with time, peaking at early 2020 (pandemic) and early 2025 (Tarrifs). Shows a need for volality to be included in the modelling. 

# Drawdown

In [11]:
market_data = calculate_drawdown(market_data)

In [12]:
#For each sector, plotting the drawdown of each of the stocks in that sector. 
for sector in sectors:
    sector_data = market_data[market_data["Sector"]==sector]
    fig = px.line(sector_data, x="Date", y="Drawdown", color="Ticker", title=f"{sector} Drawdown by Stock")
    fig.show()

# Max drawdown

In [13]:
#max_drawdown
for sector in sectors:
    sector_data = market_data[market_data['Sector']==sector]
    max_drawdown = sector_data.groupby('Ticker')['Drawdown'].min().sort_values()
    print(sector)
    print('-'*30)
    display(max_drawdown)

Technology
------------------------------


Ticker
AAPL   -0.385159
Name: Drawdown, dtype: float64

# Correlation

In [14]:

for sector in sectors:
    sector_data = market_data[market_data["Sector"]==sector]
    sector_returns = sector_data.pivot(index="Date", columns='Ticker', values='Daily_Return')
    sector_correlation = sector_returns.corr()
    print(sector)
    print("-"*20)
    display(sector_correlation)

Technology
--------------------


Ticker,AAPL
Ticker,
AAPL,1.0


Energy and Financials have the strongest correlations between the stocks while consumer and healthcare have weaker correlations. 

In [15]:
print(market_data["Drawdown"].max())

display(
    market_data.loc[market_data["Drawdown"] > 0, ["Ticker", "Date", "Drawdown"]].head()
)

0.0


Price,Ticker,Date,Drawdown
